# ⚙️ Banglish Emotion Detection: Phase 3

**Feature Engineering & Baseline Models**

In [ ]:
# 1. Setup Environment
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

!pip install lightgbm -q

# Mount Drive
drive.mount('/content/drive')
sys.path.append('/content/drive/My Drive/424 Project')

from helpers.features import extract_tfidf_features, extract_bengali_lexicon_features, combine_tfidf_and_lexicon
from helpers.baselines import train_evaluate_logistic_regression, train_evaluate_svm, train_evaluate_lightgbm
from helpers.visualization import plot_baseline_comparison, plot_confusion_matrix

In [ ]:
# 2. Load Cleaned Dataset from Phase 1
df = pd.read_csv('/content/drive/My Drive/424 Project/banglish_cleaned.csv')
print(f"Loaded {len(df)} samples.")

label_names = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

In [ ]:
# 3. Train/Val/Test Split
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df['label_id'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label_id'], random_state=42
)
y_train = train_df['label_id'].values
y_val = val_df['label_id'].values
y_test = test_df['label_id'].values
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# 4. Feature Extraction
print("Extracting TF-IDF features...")
# Handle any potential NaNs generated during reading
train_texts = train_df['text_normalized'].fillna('')
val_texts = val_df['text_normalized'].fillna('')
test_texts = test_df['text_normalized'].fillna('')

X_train_tfidf, X_val_tfidf, X_test_tfidf, tfidf_w, tfidf_c = extract_tfidf_features(
    train_texts, val_texts, test_texts
)

print("Extracting Bengali Lexicon features...")
train_bengali = train_df['Bengali'].fillna('')
val_bengali = val_df['Bengali'].fillna('')
test_bengali = test_df['Bengali'].fillna('')

lex_train, lex_val, lex_test, lex_model = extract_bengali_lexicon_features(
    train_bengali, y_train, val_bengali, test_bengali
)

print("Combining features...")
X_train_comb, X_val_comb, X_test_comb = combine_tfidf_and_lexicon(
    X_train_tfidf, X_val_tfidf, X_test_tfidf, lex_train, lex_val, lex_test
)

In [ ]:
# 5. Evaluate Baselines
results = {'Model': [], 'Weighted F1': [], 'Features': []}

print("\n--- Logistic Regression ---")
lr, y_pred_lr, f1_lr, _ = train_evaluate_logistic_regression(
    X_train_tfidf, y_train, X_test_tfidf, y_test, label_names
)
print(f"F1: {f1_lr:.4f}")
results['Model'].append('Logistic Regression'); results['Weighted F1'].append(f1_lr); results['Features'].append('TF-IDF (W+C)')

print("\n--- Linear SVM ---")
svm, y_pred_svm, f1_svm, _ = train_evaluate_svm(
    X_train_tfidf, y_train, X_test_tfidf, y_test, label_names
)
print(f"F1: {f1_svm:.4f}")
results['Model'].append('Linear SVM'); results['Weighted F1'].append(f1_svm); results['Features'].append('TF-IDF (W+C)')

print("\n--- LightGBM ---")
lgbm, y_pred_lgbm, f1_lgbm, _ = train_evaluate_lightgbm(
    X_train_tfidf, y_train, X_test_tfidf, y_test, label_names
)
print(f"F1: {f1_lgbm:.4f}")
results['Model'].append('LightGBM'); results['Weighted F1'].append(f1_lgbm); results['Features'].append('TF-IDF (W+C)')

print("\n--- SVM + Lexicon ---")
svm_lex, y_pred_svm_lex, f1_svm_lex, report_lex = train_evaluate_svm(
    X_train_comb, y_train, X_test_comb, y_test, label_names
)
print(f"F1: {f1_svm_lex:.4f}")
print("\nDetailed Report:")
print(report_lex)
results['Model'].append('SVM + Lexicon'); results['Weighted F1'].append(f1_svm_lex); results['Features'].append('TF-IDF + Lexicon')

In [ ]:
# 6. Visualizations
plot_baseline_comparison(results, output_path='/content/drive/My Drive/424 Project/baseline_comparison.png')
plot_confusion_matrix(y_test, y_pred_svm_lex, label_names, output_path='/content/drive/My Drive/424 Project/confusion_matrix_baseline.png')